# 01 - Quick Start with PGC GWAS Data

This notebook demonstrates how to load and explore psychiatric GWAS summary statistics
from the PGC collection on HuggingFace (~1 billion rows across 12 disorders).


In [ ]:
# Install the package (if needed)
# !pip install -e ..

from pgc_explorer.config import PGC_DATASETS, list_disorders
from pgc_explorer.data_loader import PGCDataLoader

# List all available disorders
for key in list_disorders():
    meta = PGC_DATASETS[key]
    print(f"{meta.display_name:.<30} {meta.approx_rows:>8} rows  ({meta.hf_id})")


## Loading Data

The `PGCDataLoader` handles HuggingFace streaming and column harmonization automatically.
Columns are mapped to a canonical schema: `snp, chr, bp, a1, a2, effect, se, pval, n, maf`.


In [ ]:
loader = PGCDataLoader()

# Load top 10,000 most significant variants for schizophrenia
df = loader.load_top_hits("schizophrenia", n=10_000)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)


In [ ]:
# Basic summary statistics
print(f"Chromosomes: {sorted(df['chr'].unique())}")
print(f"P-value range: {df['pval'].min():.2e} to {df['pval'].max():.2e}")
print(f"Genome-wide significant (p < 5e-8): {(df['pval'] < 5e-8).sum()} variants")


## Loading a Specific Region

You can query a specific genomic region (e.g., the MHC region on chromosome 6):


In [ ]:
# Load the MHC region (chr6:25-35Mb) - a hotspot for psychiatric associations
mhc = loader.load_region("schizophrenia", chr=6, start=25_000_000, end=35_000_000)
print(f"Variants in MHC region: {len(mhc)}")
mhc.sort_values('pval').head()


## Comparing Disorders

Load data from multiple disorders to find shared signals:


In [ ]:
# Load top hits for two disorders
scz = loader.load_top_hits("schizophrenia", n=5000)
bip = loader.load_top_hits("bipolar", n=5000)

# Find shared SNPs
shared = set(scz['snp']) & set(bip['snp'])
print(f"Shared significant SNPs: {len(shared)}")
